1. Get control only dataset 
2. Get neuron-biased HVGs

The idea behind our HVG approach is to obtain N_HVGS_UNSPECIFIC HVGs derived from all cells, and another N_HVGS_NEURONAL HVGs that were calculated using only neurons, trying to find HVGs that would particularly helpful to improve the subclustering of neurons.

Importantly, all is done using only control samples.

The notebook benefits from access to GPU for label propagation from the ABC cells to the rest, which is done internally in the get_neuron_biased_hvgs function below

In [1]:
# No need to restart notebook when editing custom packages
%load_ext autoreload
%autoreload 2

In [16]:
from pathlib import Path

import scanpy as sc
import pandas as pd
from dotenv import dotenv_values
import numpy as np 

from hm3d_utils.clustering import get_neuron_biased_hvgs

In [18]:
CONFIG = dotenv_values("config.env") 
DATADIR = Path(CONFIG['DATADIR'])

N_HVGS_UNSPECIFIC = 3_000
N_HVGS_NEURONAL = 1_500

# next paths will be derived
DATASET = DATADIR / "4_HY" / "all_HY" / 'datasets' / f'HY_raw_embed.h5ad'
OUTFILE_CONTROL_HVG = DATADIR / "5_clustering" / "clean_control_4500.csv"
OUTFILE_CONTROL_H5AD = DATADIR / "5_clustering" / "control.h5ad"

In [ ]:
# Ensure directory exists
OUTFILE_CONTROL_HVG.parent.mkdir(parents=True, exist_ok=True)

### load data

In [11]:
adata = sc.read_h5ad(DATASET, backed="r")

### subset to control

In [12]:
adata = adata[adata.obs.IsControl == True].to_memory()

### save 

In [ ]:
# Check dtype and convert if it's an integer to avoid rounding errors during normalization
if np.issubdtype(adata.X.dtype, np.integer):
    adata.X = adata.X.astype(np.float32)

adata.write_h5ad(OUTFILE_CONTROL_H5AD)

### get HVGs

In [20]:
final_df = get_neuron_biased_hvgs(
    adata = adata,
    n_unspec = N_HVGS_UNSPECIFIC,
    n_neuron = N_HVGS_NEURONAL,
    outfile = OUTFILE_CONTROL_HVG)

In [21]:
final_df.head()

,means,variances,gene_name,highly_variable_nbatches,highly_variable_rank,variances_norm,highly_variable
Avp,3.023283,4159.832576,Avp,201,4.0,15.010615,True
Sst,1.029848,194.531981,Sst,201,7.0,12.632971,True
Ptgds,4.478166,829.387784,Ptgds,201,19.0,9.996411,True
Cartpt,0.878280,82.537018,Cartpt,201,21.0,8.774156,True
Gal,0.944361,133.785164,Gal,201,22.0,8.293560,True
